In [ ]:
import os
import pyupbit
import pandas as pd
from core.upbit import Upbit
from loader.upbit_realtimedata_loader import UpbitRealtimeDataLoader
from processing.moving_average import MovingAverageProcessing
from processing.rsi import RSIProcessing
from processing.high_point_scoring import HighPointScoringProcessing
from processing.low_point_scoring import LowPointScoringProcessing
from processing.filtering_high_points import GetHighPoints
from processing.filtering_low_points import GetLowPoints
from processing.get_trend_section import GetTrendSections
from processing.ma_200_rising import MA200RisingProcessing
from visualization.basic_price_rsi_visualization import BasicPriceWithRsiVisualization
from notification.slack_notification import SlackNotification
from notification.slack_send_history import load_slack_send_history
from notification.slack_send_history import save_slack_send_history
from notification.slack_send_history import should_send_slack_message
from notification.slack_send_history import update_slack_send_history

import importlib
import common.common as common_module
importlib.reload(common_module)
from common.common import *

# 병합 출력 설정
target_coin_filter = ['KRW-CARV']  # 필요 시 주석 처리
load_count = 400
threshold = 0.85
indecreasing_count_set = [3, 1]

# 일봉 설정
daily_interval_base = 'day1'
daily_score_band_list = [100]

# 4시간봉 설정
h4_interval_base = 'minute240'
h4_score_band_list = [30]
ma_200_rising_min_consecutive_true = 2
aligned_in_order_window_size = 8

# Slack 설정
try:
    slack = SlackNotification(
        bot_token=os.getenv('SLACK_BOT_TOKEN'),
        channel_id="C0APBAF5DPW"
    )
    slack_enabled = True
    print("✅ Slack 연결 성공")
except Exception as e:
    slack_enabled = False
    print(f"⚠️ Slack 연결 실패: {str(e)}")

slack_send_history = load_slack_send_history()

print('merge start')
tickers = pyupbit.get_tickers('KRW')

for idx, one_coin in enumerate(tickers):
    #if target_coin_filter and one_coin not in target_coin_filter:
    #    continue

    print(f'{one_coin} 시작')

    if (idx==200): break

    # -----------------------------
    # 1) 일봉 그래프 + 빨간 라인
    # -----------------------------
    upbit_daily = Upbit()
    upbit_daily.set_loader(UpbitRealtimeDataLoader(one_coin, daily_interval_base, load_count))
    upbit_daily.load()

    daily_indicator_list = [
        MovingAverageProcessing(),
        RSIProcessing(),
        HighPointScoringProcessing(daily_score_band_list),
        LowPointScoringProcessing(daily_score_band_list),
        MA200RisingProcessing(),
    ]
    upbit_daily.add_sub_indicator(daily_indicator_list)

    upbit_daily.generate_high_low_data(
        GetHighPoints(upbit_daily.data.loc[upbit_daily.data['high_score'] != 0], 'high_score', threshold),
        GetLowPoints(upbit_daily.data.loc[upbit_daily.data['low_score'] != 0], 'low_score', threshold),
    )

    low_decline_then_rise_points = []
    if len(upbit_daily.low_point_df) > 3:
        low_values = upbit_daily.low_point_df['low'].values
        low_indices = upbit_daily.low_point_df.index.tolist()

        for i in range(len(low_values) - 3):
            if low_values[i] > low_values[i + 1] > low_values[i + 2] and low_values[i + 2] < low_values[i + 3]:
                low_decline_then_rise_points.append(low_indices[i + 3])

    draw_daily = BasicPriceWithRsiVisualization()
    draw_daily.set_data(upbit_daily.data, upbit_daily.high_point_df, upbit_daily.low_point_df)
    draw_daily.make_figure(title=f'{one_coin} - DAY1')
    figure_daily = draw_daily.get_figure()

    if len(low_decline_then_rise_points) > 0:
        add_vline_to_main_figure(
            figure_daily,
            upbit_daily.low_point_df,
            'red',
            low_decline_then_rise_points,
            label_text='일봉 저점 상승 시작',
        )

    # 일봉 빨간 라인의 날짜(yyyy-mm-dd) 집계
    daily_rise_dates = set()
    if len(low_decline_then_rise_points) > 0:
        for point_idx in low_decline_then_rise_points:
            ts = pd.to_datetime(upbit_daily.low_point_df.loc[point_idx, 'timestamp_kst'])
            daily_rise_dates.add(ts.date())

    # -------------------------------------------
    # 2) 4시간봉 그래프 + 진한 노란색 + 일봉 빨간 라인 날짜
    # -------------------------------------------
    upbit_h4 = Upbit()
    upbit_h4.set_loader(UpbitRealtimeDataLoader(one_coin, h4_interval_base, load_count))
    upbit_h4.load()

    h4_indicator_list = [
        MovingAverageProcessing(),
        RSIProcessing(),
        HighPointScoringProcessing(h4_score_band_list),
        LowPointScoringProcessing(h4_score_band_list),
        MA200RisingProcessing(),
    ]
    upbit_h4.add_sub_indicator(h4_indicator_list)

    upbit_h4.generate_high_low_data(
        GetHighPoints(upbit_h4.data.loc[upbit_h4.data['high_score'] != 0], 'high_score', threshold),
        GetLowPoints(upbit_h4.data.loc[upbit_h4.data['low_score'] != 0], 'low_score', threshold),
    )

    upbit_h4.set_processor(GetTrendSections('high', 'increasing', indecreasing_count_set[0], indecreasing_count_set[1]))
    high_point_increasing_trend_section_upbit = upbit_h4.get_key_points(upbit_h4.high_point_df)

    upbit_h4.set_processor(GetTrendSections('low', 'increasing', indecreasing_count_set[0], indecreasing_count_set[1]))
    low_point_increasing_trend_section_upbit = upbit_h4.get_key_points(upbit_h4.low_point_df)

    high_point_group_start_end_upbit = []
    for i in high_point_increasing_trend_section_upbit:
        high_point_group_start_end_upbit.append([i[0], i[-1]])

    low_point_group_start_end_upbit = []
    for i in low_point_increasing_trend_section_upbit:
        low_point_group_start_end_upbit.append([i[0], i[-1]])

    high_low_increasing_trend_section_upbit, high_low_list_dict = find_overlapping_intervals(
        high_point_group_start_end_upbit,
        low_point_group_start_end_upbit,
    )

    dark_yellow_group_start_end_upbit = []
    if 'aligned_in_order' in upbit_h4.data.columns and 'ma_200_rising' in upbit_h4.data.columns:
        ma_rising_series = upbit_h4.data['ma_200_rising'].fillna(False).astype(bool)
        ma_rising_two_consecutive = ma_rising_series
        for shift_step in range(1, ma_200_rising_min_consecutive_true):
            ma_rising_two_consecutive = ma_rising_two_consecutive & ma_rising_series.shift(shift_step, fill_value=False)

        aligned_series = upbit_h4.data['aligned_in_order'].fillna(False).astype(bool)
        aligned_all_true_in_8 = aligned_series.rolling(
            window=aligned_in_order_window_size,
            min_periods=aligned_in_order_window_size,
        ).sum().eq(aligned_in_order_window_size)

        dark_yellow_mask = ma_rising_two_consecutive & aligned_all_true_in_8
        if dark_yellow_mask.any():
            dark_yellow_group_id = (dark_yellow_mask != dark_yellow_mask.shift(fill_value=False)).cumsum()
            for _, one_group_df in upbit_h4.data.loc[dark_yellow_mask].groupby(dark_yellow_group_id[dark_yellow_mask]):
                dark_yellow_group_start_end_upbit.append([one_group_df.index[0], one_group_df.index[-1]])

    # 일봉 빨간 라인 날짜를 4시간봉 인덱스로 변환
    h4_line_index_list = []
    if len(daily_rise_dates) > 0:
        h4_dates = pd.to_datetime(upbit_h4.data['timestamp_kst']).dt.date
        for one_date in sorted(daily_rise_dates):
            matched_idx = upbit_h4.data.index[h4_dates == one_date]
            if len(matched_idx) > 0:
                h4_line_index_list.append(matched_idx[0])

    # 필터: 일봉 빨간 라인 이후에 진한 노란색 정배열 구간이 있어야 출력
    has_dark_yellow_after_daily_rise = False
    if len(h4_line_index_list) > 0 and len(dark_yellow_group_start_end_upbit) > 0:
        latest_daily_rise_idx = max(h4_line_index_list)
        has_dark_yellow_after_daily_rise = any(
            start_idx > latest_daily_rise_idx
            for start_idx, _ in dark_yellow_group_start_end_upbit
        )

    if not has_dark_yellow_after_daily_rise:
        print(f'{one_coin} 스킵: 일봉 빨간 라인 이후 진한 노란색 정배열 구간 없음')
        continue

    draw_h4 = BasicPriceWithRsiVisualization()
    draw_h4.set_data(upbit_h4.data, upbit_h4.high_point_df, upbit_h4.low_point_df)
    draw_h4.make_figure(title=f'{one_coin} - 4H')
    figure_h4 = draw_h4.get_figure()

    if len(dark_yellow_group_start_end_upbit) > 0:
        add_vrect_to_main_figure(
            figure_h4,
            upbit_h4.data,
            '#b8860b',
            dark_yellow_group_start_end_upbit,
            show_interval_label=True,
            label_prefix='번 구간',
            start_label_index=1,
            show_interval_length=True,
            interval_length_separator=' : ',
        )

    if len(h4_line_index_list) > 0:
        add_vline_to_main_figure(
            figure_h4,
            upbit_h4.data,
            'red',
            h4_line_index_list,
            label_text='일봉 저점 상승 시작',
        )

    if slack_enabled:
        should_send, current_lengths, previous_lengths = should_send_slack_message(
            slack_send_history,
            one_coin,
            dark_yellow_group_start_end_upbit,
        )

        if should_send:
            try:
                slack.send_notification_with_image(
                    f"🚀 {one_coin} - 일봉 조건 충족 차트\n진한 노란색 구간 길이: {current_lengths}",
                    figure_daily,
                )
                slack.send_notification_with_image(
                    f"🚀 {one_coin} - 4시간봉 조건 충족 차트\n진한 노란색 구간 길이: {current_lengths}",
                    figure_h4,
                )
                slack_send_history = update_slack_send_history(
                    slack_send_history,
                    one_coin,
                    dark_yellow_group_start_end_upbit,
                )
                save_slack_send_history(slack_send_history)
                print(f'{one_coin} Slack 그래프 전송 완료')
            except Exception as e:
                print(f'{one_coin} Slack 그래프 전송 실패: {str(e)}')
        else:
            print(
                f"{one_coin} Slack 전송 생략: 이미 동일한 진한 노란색 구간 이력 전송됨 "
                f"(이전/현재: {previous_lengths})"
            )

    print(f'{one_coin} 일봉 출력')
    draw_daily.visualize()

    print(f'{one_coin} 4시간봉 출력')
    draw_h4.visualize()

In [ ]:
import sys
!{sys.executable} -m pip install slack-sdk requests kaleido

In [ ]:
import os

from notification.slack_notification import SlackNotification

test_channel_id = os.getenv('SLACK_CHANNEL_ID') or 'C0APBAF5DPW'
test_message = 'SlackNotification 테스트 메시지입니다.'

try:
    slack_test = SlackNotification(
        webhook_url=os.getenv('SLACK_WEBHOOK_URL'),
        bot_token=os.getenv('SLACK_BOT_TOKEN'),
        channel_id=test_channel_id,
    )
    slack_test.send_notification(test_message)
    print('✅ 테스트 메시지 전송 완료')
except Exception as e:
    print(f'❌ 테스트 메시지 전송 실패: {str(e)}')

In [ ]:
import os
os.getenv('SLACK_BOT_TOKEN')

In [ ]:
import os
print("exists:", "SLACK_BOT_TOKEN" in os.environ)
print(os.environ)